# Predicting Product Sales
## 7. Evaluation & Comparison

We evaluate our trained models on the test set.

We implement the regression metrics from scratch:
- Mean Absolute Error (MAE):
  $$\text{MAE} = \frac{1}{n} \sum_{i=1}^n |y_i - \hat{y}_i|$$
- Mean Squared Error (MSE):
  $$\text{MSE} = \frac{1}{n} \sum_{i=1}^n (y_i - \hat{y}_i)^2$$
- Root Mean Squared Error (RMSE):
  $$\text{RMSE} = \sqrt{\text{MSE}}$$
- Coefficient of Determination ($R^2$):
  $$R^2 = 1 - \frac{\sum_{i=1}^n (y_i - \hat{y}_i)^2}{\sum_{i=1}^n (y_i - \bar{y})^2}$$

In [ ]:
import numpy as np
import pandas as pd
import joblib
import time
import os
import sys
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

# Define class structure so joblib.load can deserialize the scratch model
class LightGBMFromScratch:
    def __init__(self, n_estimators=50, learning_rate=0.1, max_depth=3, max_leaf_nodes=8, n_bins=32):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.max_leaf_nodes = max_leaf_nodes
        self.n_bins = n_bins
        self.base_pred = None
        self.trees = []
        self.bin_edges = {}

    def _bin_features(self, X, fit=False):
        X_binned = X.copy()
        for col in X.columns:
            if fit:
                n_unique = X[col].nunique()
                if n_unique > self.n_bins:
                    percentiles = np.linspace(0, 100, self.n_bins + 1)
                    edges = np.percentile(X[col], percentiles)
                    edges = np.unique(edges)
                    self.bin_edges[col] = edges
                else:
                    self.bin_edges[col] = None
            edges = self.bin_edges[col]
            if edges is not None:
                X_binned[col] = np.digitize(X[col], edges[1:-1])
        return X_binned

    def predict(self, X):
        X_binned = self._bin_features(X, fit=False)
        preds = np.full(len(X), self.base_pred)
        for tree in self.trees:
            preds += self.learning_rate * tree.predict(X_binned)
        return preds

# Register in main module so pickler loads it properly
sys.modules['__main__'].LightGBMFromScratch = LightGBMFromScratch

READY_DATA_DIR = r'./data/ready_for_train'
X_test = joblib.load(os.path.join(READY_DATA_DIR, 'X_test.pkl'))
y_test = joblib.load(os.path.join(READY_DATA_DIR, 'y_test.pkl'))
fit_times = joblib.load(os.path.join(READY_DATA_DIR, 'fit_times.pkl'))

print('Test set size:', X_test.shape)

### 7.1 Custom Metrics Implementation

In [ ]:
def compute_mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def compute_mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def compute_rmse(y_true, y_pred):
    return np.sqrt(compute_mse(y_true, y_pred))

def compute_r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1.0 - (ss_res / ss_tot) if ss_tot > 0 else 0.0

### 7.2 Evaluate and Compare Models
We compare official libraries with our custom-built **LightGBM (Scratch)**.

In [ ]:
model_names = ['HistGradientBoosting', 'XGBoost', 'LightGBM', 'CatBoost', 'LightGBM (Scratch)']
results = []
predictions = {}

for name in model_names:
    clean_name = name.lower().replace(' ', '_').replace('(', '').replace(')', '')
    model_path = os.path.join(READY_DATA_DIR, f'model_{clean_name}.pkl')
    model = joblib.load(model_path)
    
    start_time = time.time()
    y_pred = model.predict(X_test)
    pred_time = time.time() - start_time
    
    predictions[name] = y_pred
    
    mae = compute_mae(y_test.values, y_pred)
    rmse = compute_rmse(y_test.values, y_pred)
    r2 = compute_r2(y_test.values, y_pred)
    
    results.append({
        'Model': name,
        'MAE': mae,
        'RMSE': rmse,
        'R2 Score': r2,
        'Fit Time (s)': fit_times[name],
        'Predict Time (ms)': pred_time * 1000
    })

df_results = pd.DataFrame(results)
print(df_results.sort_values(by='R2 Score', ascending=False).to_string(index=False))

### 7.3 Visualize Performance Metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

df_sorted_r2 = df_results.sort_values(by='R2 Score', ascending=True)
sns.barplot(data=df_sorted_r2, x='R2 Score', y='Model', ax=axes[0], palette='viridis')
axes[0].set_title('R-squared Score Comparison (Higher is Better)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('R2 Score')

df_sorted_rmse = df_results.sort_values(by='RMSE', ascending=False)
sns.barplot(data=df_sorted_rmse, x='RMSE', y='Model', ax=axes[1], palette='magma')
axes[1].set_title('RMSE Comparison (Lower is Better)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('RMSE')

plt.tight_layout()
plt.show()

### 7.4 Residual Analysis (Best Model)

In [ ]:
best_model_name = df_results.loc[df_results['R2 Score'].idxmax(), 'Model']
print('Best performing model:', best_model_name)

best_preds = predictions[best_model_name]
residuals = y_test.values - best_preds

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(y_test.values, best_preds, alpha=0.3, color='teal')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--', color='red', linewidth=2)
axes[0].set_title(f'{best_model_name}: Predictions vs Actuals', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Actual Sales Revenue')
axes[0].set_ylabel('Predicted Sales Revenue')

axes[1].scatter(best_preds, residuals, alpha=0.3, color='crimson')
axes[1].axhline(y=0, color='black', linestyle='--', linewidth=2)
axes[1].set_title(f'{best_model_name}: Residuals Plot', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Predicted Sales Revenue')
axes[1].set_ylabel('Residuals')

plt.tight_layout()
plt.show()